# From PCA to Deep Autoencoders with Keras

An autoencoder compresses its input into a small code (the encoder), then rebuilds it from that code (the decoder): its target is its own input. This notebook trains several of them on MNIST (60,000 training and 10,000 test images of handwritten digits, 28 × 28 in grayscale), from the simplest to the deepest:

1. A linear autoencoder
2. The linear autoencoder against PCA
3. A deep autoencoder
4. The latent space
5. Anomaly detection
6. The code size
7. Going further: a sparse autoencoder

Each part reuses the models of the previous ones; the last one is optional. No GPU needed: the notebook runs in about twenty minutes on a CPU, half of it for part 6, which trains six autoencoders.

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import layers
from sklearn.decomposition import PCA

keras.utils.set_random_seed(0)

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
input_dim = 28 * 28
# Pixels in [0, 1], like a sigmoid's output: neither centered nor scaled
X_train = X_train.reshape(-1, input_dim) / 255
X_test = X_test.reshape(-1, input_dim) / 255
print(X_train.shape, X_test.shape)


def show_images(*rows: np.ndarray, titles: list | np.ndarray | None = None) -> None:
  # Shows rows of 28 × 28 images, one row per array
  n = len(rows[0])
  _, axes = plt.subplots(
    len(rows), n, figsize=(n * 1.2, len(rows) * 1.3), squeeze=False
  )
  for row, images in zip(axes, rows):
    for ax, image in zip(row, images):
      ax.imshow(np.reshape(image, (28, 28)), cmap="gray_r")
      ax.axis("off")
  for ax, title in zip(axes[0], [] if titles is None else titles):
    ax.set_title(title)
  plt.show()


show_images(X_train[:10], titles=y_train[:10])

## 1. A linear autoencoder

Without an activation function, the encoder and the decoder are only linear maps (plus a bias). Build, as on the "Autoencoder in Keras" slide but without activations:

- `encoder`: a `Dense` layer of `encoding_dim = 20` units, on inputs of size `input_dim`;
- `decoder`: a `Dense` layer of `input_dim` units, on codes of size `encoding_dim`;
- `autoencoder`, which chains the two, compiled with the Adam optimizer and the `"mse"` loss.

The cell after the solution trains it.

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `keras.Input(shape=(input_dim,))` at the head of each `keras.Sequential`, which sets the size of its inputs
- `layers.Dense(n)` without an `activation` argument: a linear layer
- `keras.Sequential([encoder, decoder], name="autoencoder")`: a model can serve as a layer of another one, and `encoder` and `decoder` stay usable on their own
- `autoencoder.summary()` should show 15,700 parameters for the encoder (784 × 20 weights + 20 biases) and 16,464 for the decoder

A skeleton to copy into the exercise cell, replacing each `...`:

```python
encoding_dim = 20
encoder = keras.Sequential(
  [keras.Input(shape=(input_dim,)), layers.Dense(...)], name="encoder"
)
decoder = keras.Sequential([..., ...], name="decoder")
autoencoder = keras.Sequential([..., ...], name="autoencoder")
autoencoder.compile(optimizer=..., loss=...)
autoencoder.summary()
```

### Solution

In [ ]:
encoding_dim = 20
encoder = keras.Sequential(
  [keras.Input(shape=(input_dim,)), layers.Dense(encoding_dim)], name="encoder"
)
decoder = keras.Sequential(
  [keras.Input(shape=(encoding_dim,)), layers.Dense(input_dim)], name="decoder"
)
autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

### Training

The target is the input: `fit(X_train, X_train)`. The validation loss is computed on the test images.

In [ ]:
autoencoder.fit(
  X_train,
  X_train,
  epochs=10,
  batch_size=256,
  validation_data=(X_test, X_test),
  verbose=2,
)
linear_decoder = decoder  # kept for part 2
X_test_linear = autoencoder.predict(X_test, verbose=0)
show_images(X_test[:10], X_test_linear[:10], titles=y_test[:10])

## 2. The linear autoencoder against PCA

The "Linear autoencoder and PCA" slide states it: trained with the MSE, a linear autoencoder with a code of size k finds the subspace of PCA's first k components, but not its axes. Check it:

- fit a `PCA` with `encoding_dim` components on `X_train`, rebuild `X_test` with it, and compare the mean squared error of its reconstruction with the autoencoder's (`X_test_linear`, computed by the previous cell);
- the decoder's 20 directions are the rows of its kernel `W` (20 × 784): measure, for each one, the share of its norm that lies in PCA's subspace;
- finally, measure the largest absolute cosine between two directions of the decoder, then between two components of PCA.

Are the two errors close? Do the decoder's directions lie in PCA's subspace? Are they orthogonal, like the components?

**Useful documentation**

- [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [`Layer.get_weights`](https://keras.io/api/layers/base_layer/#getweights-method)
- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `pca = PCA(encoding_dim).fit(X_train)`, then `pca.inverse_transform(pca.transform(X_test))`: PCA's reconstruction; the error: `np.mean((X_test - X_rec) ** 2)`
- `W = linear_decoder.layers[0].get_weights()[0]`: the kernel of the decoder's single `Dense` layer, of shape (20, 784), one direction per row (`get_weights()[1]` is the bias)
- the rows of `pca.components_` (20 × 784) are orthonormal: `W @ pca.components_.T` gives each direction's coordinates in PCA's basis; the norm of each row of that product (`np.linalg.norm(..., axis=1)`), divided by the norm of the row of `W`, is the share you are looking for (1: the direction lies entirely in the subspace)
- the cosines: normalize the rows, `V / np.linalg.norm(V, axis=1, keepdims=True)`, then `V @ V.T`; off the diagonal (`~np.eye(20, dtype=bool)`), 0 for two orthogonal directions

A skeleton to copy into the exercise cell, replacing each `...`:

```python
pca = PCA(...).fit(...)
X_test_pca = ...
pca_error = np.mean((X_test - X_test_pca) ** 2)
linear_error = ...
print(f"Error: PCA {pca_error:.4f}, linear AE {linear_error:.4f}")

W = linear_decoder.layers[0].get_weights()[0]
share = ...
print(f"Share in PCA's subspace: {share.min():.3f} to {share.max():.3f}")


def max_cosine(V: np.ndarray) -> float:
  V = ...
  cosines = ...
  return np.abs(cosines[~np.eye(len(V), dtype=bool)]).max()


print(f"Largest cosine: decoder {max_cosine(W):.2f}, PCA {max_cosine(...):.2f}")
```

### Solution

In [ ]:
pca = PCA(encoding_dim).fit(X_train)
X_test_pca = pca.inverse_transform(pca.transform(X_test))
pca_error = np.mean((X_test - X_test_pca) ** 2)
linear_error = np.mean((X_test - X_test_linear) ** 2)
print(f"Error: PCA {pca_error:.4f}, linear AE {linear_error:.4f}")

W = linear_decoder.layers[0].get_weights()[0]  # (20, 784)
share = np.linalg.norm(W @ pca.components_.T, axis=1) / np.linalg.norm(W, axis=1)
print(f"Share in PCA's subspace: from {share.min():.3f} to {share.max():.3f}")


def max_cosine(V: np.ndarray) -> float:
  V = V / np.linalg.norm(V, axis=1, keepdims=True)
  cosines = V @ V.T
  return np.abs(cosines[~np.eye(len(V), dtype=bool)]).max()


print(
  f"Largest cosine: decoder {max_cosine(W):.2f}, PCA {max_cosine(pca.components_):.2f}"
)

The two errors are nearly equal: the linear autoencoder does as well as PCA, but no better, since PCA gives the best possible linear reconstruction with 20 dimensions. Its 20 directions lie almost entirely in PCA's subspace (over 95%): it found the same subspace, up to the training's inaccuracies. But they aren't orthogonal (the largest cosine is about 0.2, against 0 for PCA), nor sorted by explained variance: nothing in the loss requires it. PCA gives that basis on top, and without any training.

## 3. A deep autoencoder

Add a nonlinear hidden layer on each side of the code, as on the "Autoencoder in Keras" slide, and leave the output choice open: write `build_autoencoder(encoding_dim, output_activation, loss)`, which returns the compiled autoencoder:

- `encoder`: a `Dense` of 128 units, then one of `encoding_dim` units, both with the ReLU activation;
- `decoder`: a `Dense` of 128 units (ReLU), then the output layer, of `input_dim` units and activation `output_activation` (`None`: linear output);
- the autoencoder, which chains the two, compiled with the Adam optimizer and the loss `loss`.

The cell after the solution trains two of them, with part 1's code of 20 numbers: one with a linear output and the `"mse"` loss, the other with a sigmoid output and the `"binary_crossentropy"` loss. It compares their mean squared errors, the only measure both losses share, with PCA's and the linear autoencoder's, and gives the range of the pixels each one returns. Which one suits pixels in [0, 1], and why?

In [ ]:
def build_autoencoder(
  encoding_dim: int, output_activation: str | None, loss: str
) -> keras.Model:
  # Your code here
  ...

### Additional hints

The calls and parameters to use:

- `layers.Dense(128, activation="relu")`: a hidden layer; `layers.Dense(input_dim, activation=output_activation)`: the output, linear when `output_activation` is `None`
- `keras.Sequential([encoder, decoder], name="autoencoder")`, then `autoencoder.compile(optimizer="adam", loss=loss)` and `return autoencoder`
- `build_autoencoder(20, "sigmoid", "binary_crossentropy").summary()` must show 103,060 parameters for the encoder, 103,824 for the decoder

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def build_autoencoder(
  encoding_dim: int, output_activation: str | None, loss: str
) -> keras.Model:
  encoder = keras.Sequential(
    [
      keras.Input(shape=(input_dim,)),
      layers.Dense(128, activation="relu"),
      ...,
    ],
    name="encoder",
  )
  decoder = keras.Sequential(
    [keras.Input(shape=(encoding_dim,)), ..., layers.Dense(input_dim, activation=...)],
    name="decoder",
  )
  autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
  autoencoder.compile(...)
  return autoencoder
```

### Solution

In [ ]:
def build_autoencoder(
  encoding_dim: int, output_activation: str | None, loss: str
) -> keras.Model:
  encoder = keras.Sequential(
    [
      keras.Input(shape=(input_dim,)),
      layers.Dense(128, activation="relu"),
      layers.Dense(encoding_dim, activation="relu"),
    ],
    name="encoder",
  )
  decoder = keras.Sequential(
    [
      keras.Input(shape=(encoding_dim,)),
      layers.Dense(128, activation="relu"),
      # None: linear output; "sigmoid": output in ]0, 1[
      layers.Dense(input_dim, activation=output_activation),
    ],
    name="decoder",
  )
  autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
  autoencoder.compile(optimizer="adam", loss=loss)
  return autoencoder

### Training and comparison

Two deep autoencoders, 10 epochs each: the cell takes a few minutes.

In [ ]:
X_test_deep = {}
for output_activation, loss in [(None, "mse"), ("sigmoid", "binary_crossentropy")]:
  autoencoder = build_autoencoder(encoding_dim, output_activation, loss)
  autoencoder.fit(
    X_train,
    X_train,
    epochs=10,
    batch_size=256,
    validation_data=(X_test, X_test),
    verbose=0,
  )
  X_test_pred = autoencoder.predict(X_test, verbose=0)
  X_test_deep[loss] = X_test_pred
  print(
    f"Output {output_activation or 'linear'}, loss {loss}: "
    f"error {np.mean((X_test - X_test_pred) ** 2):.4f}, "
    f"pixels from {X_test_pred.min():.2f} to {X_test_pred.max():.2f}"
  )
print(f"Error: PCA {pca_error:.4f}, linear AE {linear_error:.4f}")
# At the top the originals, then the linear AE, then the deep AEs (MSE, then BCE)
show_images(
  X_test[:10],
  X_test_linear[:10],
  X_test_deep["mse"][:10],
  X_test_deep["binary_crossentropy"][:10],
  titles=y_test[:10],
)
# What follows keeps the last one, with a sigmoid output, and its two halves
encoder, decoder = autoencoder.layers

With the same code of 20 numbers, the deep autoencoder with a sigmoid output rebuilds much better than PCA and the linear autoencoder: its nonlinear layers follow curved surfaces, out of a subspace's reach. The one with a linear output does barely better than PCA: it returns pixels outside [0, 1], below 0 and above 1, and a gray background where the image is white, values the sigmoid rules out by design. For pixels in [0, 1], the sigmoid output is the one to use; with it, `binary_crossentropy`, the usual choice, which treats each pixel as a probability, or the MSE. What follows keeps this autoencoder, in `autoencoder`, `encoder` and `decoder`.

## 4. The latent space

The encoder alone turns each test image into its code, a point of a space with `encoding_dim` dimensions: the latent space.

In [ ]:
codes = encoder.predict(X_test, verbose=0)
print(codes.shape)

### Where do the codes live?

Could the decoder alone make new digits from random codes? First look at where the real codes live, then try:

- for each of the 20 dimensions, print the minimum, the mean and the maximum of `codes`, and count the dimensions that are 0 for every image;
- decode 10 white noise codes, drawn uniformly in [0, 1] on each dimension, then 10 codes drawn uniformly between each dimension's minimum and maximum, and show both rows of images with `show_images`.

Do the codes look like uniform noise in [0, 1]? Do the decoded images look like digits? Why?

**Useful documentation**

- [`numpy.random.Generator.uniform`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.uniform.html)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `codes.min(axis=0)`, `codes.mean(axis=0)`, `codes.max(axis=0)`: one value per dimension; `np.round(..., 1)` to print them
- a dimension that is 0 for every image: `codes.max(axis=0) == 0` (the code's ReLU never returns less than 0), then `.sum()`
- `rng = np.random.default_rng(0)`, then `rng.uniform(0, 1, size=(10, encoding_dim))`; `rng.uniform(low, high, size=(10, encoding_dim))` also accepts arrays `low` and `high`, one value per dimension
- `decoder.predict(noise, verbose=0)` returns 10 images of 784 pixels, which `show_images` shows

A skeleton to copy into the exercise cell, replacing each `...`:

```python
low, high = ..., ...
print("Min:", np.round(low, 1))
print("Mean:", ...)
print("Max:", ...)
print("Dead dimensions:", int(...))

rng = np.random.default_rng(0)
white_noise = rng.uniform(...)
in_range = rng.uniform(...)
show_images(decoder.predict(white_noise, verbose=0), decoder.predict(..., verbose=0))
```

#### Solution

In [ ]:
low, high = codes.min(axis=0), codes.max(axis=0)
print("Min:", np.round(low, 1))
print("Mean:", np.round(codes.mean(axis=0), 1))
print("Max:", np.round(high, 1))
print("Dead dimensions:", int((high == 0).sum()))

rng = np.random.default_rng(0)
white_noise = rng.uniform(0, 1, size=(10, encoding_dim))
in_range = rng.uniform(low, high, size=(10, encoding_dim))
# Top: white noise, bottom: codes drawn in their range
show_images(
  decoder.predict(white_noise, verbose=0), decoder.predict(in_range, verbose=0)
)

The codes don't look like uniform noise in [0, 1]: the ReLU makes them positive or zero, their range changes from one dimension to the next and reaches several tens, and one dimension or more may be dead, zero for every image. White noise thus falls into a corner of the latent space, near the origin, that the decoder has almost never seen: it returns blots that all look alike. Drawn in their range, the codes do no better: the real codes take up only a small part of that box, as their dimensions are tied together. An autoencoder compresses, but doesn't generate: its decoder can only decode the codes the encoder produces. The variational autoencoder, in the next notebook, forces the codes to follow a known distribution, $\mathcal{N}(0, I)$, from which one can draw.

### The digits' centroids

The codes aren't scattered at random: those of the same digit cluster together. The mean code of each digit, its centroid, once decoded, gives a typical image of it.

In [ ]:
centroids = np.stack([codes[y_test == digit].mean(axis=0) for digit in range(10)])
show_images(decoder.predict(centroids, verbose=0), titles=list(range(10)))

### Interpolating between two centroids

Codes evenly spaced on the segment between two centroids, once decoded, show how the decoder goes from one digit to the other. Try other pairs.

In [ ]:
def latent_walk(start: int, end: int, n: int = 12) -> None:
  # n evenly spaced codes, from start's centroid to end's
  walk = np.linspace(centroids[start], centroids[end], n)
  show_images(decoder.predict(walk, verbose=0))


latent_walk(7, 1)
latent_walk(3, 8)

## 5. Anomaly detection

An autoencoder rebuilds well what looks like its training examples, and badly the rest (the "Anomaly detection by reconstruction error" slide). Yours has only seen digits: the next cell gives it anomalies, 1,000 images of clothes from Fashion-MNIST, in `X_anomalies`. The threshold is set on normal images the model didn't learn: the test images, which `fit` only uses to show the validation loss, cut in two halves, every other image, `X_val` to set the threshold and `X_normal` to put it to the test (two contiguous halves don't look enough alike: MNIST's images aren't in random order). The cell rebuilds them all, into `X_val_pred`, `X_normal_pred` and `X_anomalies_pred`. Then:

- compute the mean squared reconstruction error of each image, into `val_errors`, `normal_errors` and `anomaly_errors`;
- set the threshold at the 99% quantile of `val_errors`, and give the share of the normal images of `X_normal`, then that of the anomalies, above it;
- show, above their reconstruction, the 10 worst rebuilt images of `X_normal`, the 10 best rebuilt ones, then 10 anomalies.

Which share of the anomalies does the threshold catch, and which share of the normal images does it flag wrongly? Are the worst rebuilt normal images badly written digits, or digits of one particular class? And the best rebuilt ones?

**Useful documentation**

- [`numpy.quantile`](https://numpy.org/doc/stable/reference/generated/numpy.quantile.html)
- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# The normal ones: the test images, which fit didn't learn, every other one each side
X_val, X_normal = X_test[::2], X_test[1::2]
y_normal = y_test[1::2]
# The anomalies: 1,000 Fashion-MNIST images, clothes
(X_fashion, _), _ = keras.datasets.fashion_mnist.load_data()
X_anomalies = X_fashion[:1000].reshape(-1, input_dim) / 255
X_val_pred = autoencoder.predict(X_val, verbose=0)
X_normal_pred = autoencoder.predict(X_normal, verbose=0)
X_anomalies_pred = autoencoder.predict(X_anomalies, verbose=0)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `np.mean((X_val - X_val_pred) ** 2, axis=1)`: one error per image (the mean over its 784 pixels)
- `np.quantile(val_errors, 0.99)`: the value that 1% of the errors of `X_val` exceed; `(normal_errors > threshold).mean()`: the share of the images of `X_normal` above it, about 1% if they look like those of `X_val`
- `np.argsort(normal_errors)`: the indices of the images, from the smallest error to the largest; `[::-1]` reverses the order
- `show_images(X_normal[worst], X_normal_pred[worst], titles=y_normal[worst])`: the originals above their reconstruction

A skeleton to copy into the exercise cell, replacing each `...`:

```python
val_errors = ...
normal_errors = ...
anomaly_errors = ...
threshold = ...
print(f"Normal images above it: {...:.1%}, anomalies above it: {...:.1%}")

order = np.argsort(normal_errors)
worst, best = order[::-1][:10], ...
show_images(X_normal[worst], X_normal_pred[worst], titles=y_normal[worst])
show_images(...)
show_images(X_anomalies[:10], ...)
```

### Solution

In [ ]:
val_errors = np.mean((X_val - X_val_pred) ** 2, axis=1)
normal_errors = np.mean((X_normal - X_normal_pred) ** 2, axis=1)
anomaly_errors = np.mean((X_anomalies - X_anomalies_pred) ** 2, axis=1)
threshold = np.quantile(val_errors, 0.99)
print(f"Threshold: {threshold:.4f}")
print(f"Normal images above it: {(normal_errors > threshold).mean():.1%}")
print(f"Anomalies above it: {(anomaly_errors > threshold).mean():.1%}")

order = np.argsort(normal_errors)
worst, best = order[::-1][:10], order[:10]
print("The worst rebuilt normal images")
show_images(X_normal[worst], X_normal_pred[worst], titles=y_normal[worst])
print("The best rebuilt ones")
show_images(X_normal[best], X_normal_pred[best], titles=y_normal[best])
print("Some anomalies")
show_images(X_anomalies[:10], X_anomalies_pred[:10])

About 1% of the normal images of `X_normal` exceed the threshold, as expected: `X_val` and `X_normal` come from the same data. The vast majority of the anomalies exceed it, nearly 9 out of 10: the autoencoder has never seen clothes, and rebuilds them as blobs pulled towards a digit. The threshold doesn't catch them all: lowering it would catch more, at the cost of more false alarms. The worst rebuilt normal images are digits of a rare shape, thick or ill-formed: they stray from what the autoencoder has seen, and its reconstruction pulls them towards a more common digit. The best rebuilt ones are mostly 1s: few lit pixels, a simple shape, hence little error. So the error also depends on the class: with a single threshold, a complicated digit is flagged more often than a 1.

## 6. The code size

A larger code rebuilds better, but compresses less (the "Undercomplete or overcomplete code" slide). Measure that trade-off, with `build_autoencoder`:

- train an autoencoder with a sigmoid output and the `"binary_crossentropy"` loss for each code size of 2, 4, 8, 16, 32 and 64, for 10 epochs each, with the same `fit` as in part 3;
- keep each trained model in a dictionary `autoencoders`, and its last validation loss in `val_losses`, keyed by the code size;
- plot the validation loss against the code size, with a base-2 logarithmic x axis;
- show the first 10 test images, then their reconstructions by each model, one row per code size.

From which size do the rebuilt digits become sharp? Which code size would you pick, and which compression ratio (784 / code size) do you get then?

**Useful documentation**

- [`Model.fit` (returned value: `History`)](https://keras.io/api/models/model_training_apis/#fit-method)
- [`matplotlib.pyplot.xscale`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.xscale.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `build_autoencoder(size, "sigmoid", "binary_crossentropy")`: part 3's autoencoder, for a code of `size` numbers
- `history = autoencoders[size].fit(...)`: `history.history["val_loss"]` is the list of validation losses, one per epoch, and `[-1]` the last one
- the loop takes a few minutes: `print` each size's loss as you go
- `plt.plot(list(val_losses), list(val_losses.values()), marker="o")`, then `plt.xscale("log", base=2)`
- `show_images(X_test[:10], *rebuilt)`: `*` passes each item of the list `rebuilt`, one reconstruction per model, as a row; `model(X_test[:10])` calls the model directly, faster than `predict` for 10 images, and `keras.ops.convert_to_numpy` turns the result into a NumPy array

A skeleton to copy into the exercise cell, replacing each `...`:

```python
autoencoders, val_losses = {}, {}
for size in [2, 4, 8, 16, 32, 64]:
  autoencoders[size] = build_autoencoder(...)
  history = autoencoders[size].fit(...)
  val_losses[size] = ...
  print(f"Code of size {size}: {val_losses[size]:.4f}")

plt.plot(...)
plt.xscale(...)
plt.xlabel("Code size")
plt.ylabel("Validation loss")
plt.show()

rebuilt = [keras.ops.convert_to_numpy(model(...)) for model in ...]
show_images(X_test[:10], *rebuilt, titles=y_test[:10])
```

### Solution

In [ ]:
autoencoders, val_losses = {}, {}
for size in [2, 4, 8, 16, 32, 64]:
  autoencoders[size] = build_autoencoder(size, "sigmoid", "binary_crossentropy")
  history = autoencoders[size].fit(
    X_train,
    X_train,
    epochs=10,
    batch_size=256,
    validation_data=(X_test, X_test),
    verbose=0,
  )
  val_losses[size] = history.history["val_loss"][-1]
  print(
    f"Code of size {size} (compression {input_dim / size:.0f}x): "
    f"validation loss {val_losses[size]:.4f}"
  )

plt.plot(list(val_losses), list(val_losses.values()), marker="o")
plt.xscale("log", base=2)
plt.xlabel("Code size")
plt.ylabel("Validation loss")
plt.show()

# At the top the originals, then one row per code size, from 2 to 64
rebuilt = [
  keras.ops.convert_to_numpy(model(X_test[:10])) for model in autoencoders.values()
]
show_images(X_test[:10], *rebuilt, titles=y_test[:10])

The loss drops fast while the code is very small, then less and less: each doubling of the code brings less than the previous one. The images show it: with 2 or 4 numbers, the reconstructions are blurry and several digits get mixed up, a 4 or a 5 turning into a 9; from 16 on, they are sharp. A code of 16 to 32 dimensions is a good trade-off, a compression ratio of 784 / 32 = 24.5 to 784 / 16 = 49 (in number of values: the code and the pixels are all floats).

### The two-dimensional code

With a code of 2 numbers, the latent space of `autoencoders[2]` can be drawn: each test image becomes a point in it, colored by its digit, a label the autoencoder has never seen (the "Latent space of an autoencoder" slide). Next to it, the same images projected on the first two components of PCA, part 2's linear dimension reduction.

In [ ]:
codes_2d = autoencoders[2].layers[0].predict(X_test, verbose=0)
pca_2d = PCA(2).fit(X_train).transform(X_test)
_, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, points, title in [
  (axes[0], codes_2d, "Autoencoder, code of 2 numbers"),
  (axes[1], pca_2d, "PCA, 2 components"),
]:
  for digit in range(10):
    selected = points[y_test == digit]
    ax.scatter(selected[:, 0], selected[:, 1], s=2, alpha=0.5, label=digit)
  ax.set_title(title)
axes[1].legend(markerscale=5, loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

Without ever seeing a label, the autoencoder isolates some digits better than PCA: the 1s, for instance, form a branch of their own. But the ReLU of its code crushes many codes to 0, on one axis or both: the other digits pile up near the origin, where they mix more than in PCA. The codes also leave large gaps, where a point picked at random doesn't decode into a digit. A code without an activation, like the one of the "Latent space of an autoencoder" slide, spreads out better.

## 7. Going further: a sparse autoencoder

A sparse autoencoder usually has a code wider than the input, but a penalty on its activations leaves only a few of them active for each image; here we keep 64 units so that training stays short, the penalty acts the same way. Build `sparse_autoencoder`:

- the deep autoencoder, with a code of 64 units, followed by an `L1Penalty` layer that adds to the loss an L1 penalty of coefficient 1e-3 on its activations (the sum of their absolute values for each image, averaged over the images of the batch);
- train it for 10 epochs, with the same `fit` as in part 6.

Compare it with part 6's autoencoder with a 64-unit code, `autoencoders[64]`: the mean number of active (> 0) code units per test image, the number of units never active, and the mean squared reconstruction error. What does sparsity cost?

**Useful documentation**

- [Losses added by a layer, `add_loss`](https://keras.io/api/layers/base_layer/#addloss-method)
- [Keras layer regularizers](https://keras.io/api/layers/regularizers/)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `layers.Dense(64, activation="relu")`, then `L1Penalty(1e-3)`: the code's layer, followed by the penalty
- `self.add_loss(self.coef * keras.ops.mean(keras.ops.sum(keras.ops.abs(x), axis=1)))`: in `call`, the layer's penalty, the sum of the absolute values of each image's 64 activations (`axis=1`), averaged over the images of the batch; `call` then returns `x` unchanged
- `activity_regularizer=keras.regularizers.L1(...)`, Keras layers' argument for such a penalty, changed scale with Keras 3.14, which divides it by the batch size: the same coefficient doesn't give the same code from one version to the next, hence the hand-written layer
- `autoencoders[64].layers[0]`: the encoder of part 6's autoencoder (the first model of its `Sequential`)
- `codes = encoder.predict(X_test, verbose=0)`, then `(codes > 0).sum(axis=1).mean()`: the mean number of active units per image; `(codes.max(axis=0) == 0).sum()`: the units never active

A skeleton to copy into the exercise cell, replacing each `...`:

```python
class L1Penalty(layers.Layer):
  def __init__(self, coef: float, **kwargs) -> None:
    super().__init__(**kwargs)
    self.coef = coef

  def call(self, x):
    self.add_loss(...)
    return x


sparse_encoder = keras.Sequential(
  [
    keras.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    L1Penalty(...),
  ]
)
sparse_decoder = ...
sparse_autoencoder = keras.Sequential([sparse_encoder, sparse_decoder])
sparse_autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
sparse_autoencoder.fit(...)

for name, model in [("No penalty", autoencoders[64]), ("L1", sparse_autoencoder)]:
  codes = model.layers[0].predict(X_test, verbose=0)
  error = np.mean((X_test - model.predict(X_test, verbose=0)) ** 2)
  print(f"{name}: ...")
```

### Solution

In [ ]:
class L1Penalty(layers.Layer):
  # Returns its input unchanged, and adds to the loss coef times the sum of the
  # absolute values of each image's activations, averaged over the batch
  def __init__(self, coef: float, **kwargs) -> None:
    super().__init__(**kwargs)
    self.coef = coef

  def call(self, x):
    self.add_loss(self.coef * keras.ops.mean(keras.ops.sum(keras.ops.abs(x), axis=1)))
    return x


sparse_encoder = keras.Sequential(
  [
    keras.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    L1Penalty(1e-3),
  ],
  name="sparse_encoder",
)
sparse_decoder = keras.Sequential(
  [
    keras.Input(shape=(64,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(input_dim, activation="sigmoid"),
  ],
  name="sparse_decoder",
)
sparse_autoencoder = keras.Sequential(
  [sparse_encoder, sparse_decoder], name="sparse_autoencoder"
)
sparse_autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
sparse_autoencoder.fit(
  X_train,
  X_train,
  epochs=10,
  batch_size=256,
  validation_data=(X_test, X_test),
  verbose=0,
)

for name, model in [("No penalty", autoencoders[64]), ("L1", sparse_autoencoder)]:
  codes = model.layers[0].predict(X_test, verbose=0)
  error = np.mean((X_test - model.predict(X_test, verbose=0)) ** 2)
  print(
    f"{name}: {(codes > 0).sum(axis=1).mean():.1f} active units per image out of 64, "
    f"{(codes.max(axis=0) == 0).sum()} never active, error {error:.4f}"
  )

With the penalty, each image activates only about twenty code units instead of about fifty, and about thirty of the 64 units are never active: the penalty switches off the least useful units. Sparsity is paid for in reconstruction, whose error nearly doubles. The coefficient sets that trade-off: at 1e-4, the effect is much weaker, about fifty units staying active; at 1e-2, the code almost switches off, one active unit per image on average. In a sparse code, each active unit carries a pattern one can look at on its own: that is what serves to interpret the representations of large language models.